### Categories Parsing:

In [ ]:
import json
import math

In [ ]:
classes = ['bulkcarrier', 'containership', 'generalcargo', 'tanker', 'other']
classes_dict = {elem:idx for idx,elem in enumerate(classes)}
print(classes_dict)

In [ ]:
def get_coco(annotation_file):
    with open(annotation_file, 'r') as f:
        data = json.load(f)
    return data

def save_to_json(data, file_path):
    """
    Saves the given data to a JSON file specified by file_path.

    Parameters:
    data (dict): The data to be saved. This should be a dictionary.
    file_path (str): The file path where the JSON file will be saved.

    Returns:
    None
    """
    try:
        with open(file_path, 'w') as file:
            json.dump(data, file, indent=4)  # Using indent for better readability
        print("Data successfully saved to", file_path)
    except Exception as e:
        print("Failed to save data:", e)

def remapper(class_type: str):
    """Function to remap the vessel type in the 5 original classes

    Args:
        class_type (str): The class reported in the AIS file
    """
    classes_dict = {elem:idx for idx,elem in enumerate(classes)}
    
    if 'Carrier' in class_type:
        class_type = 'bulkcarrier'
    elif 'Cargo' in class_type:
        class_type = 'generalcargo'
    elif 'Container' in class_type:
        class_type = 'containership'
    elif 'Tanker' in class_type:
        class_type = 'tanker'
    else:
        class_type = 'other'
    
    return classes_dict[class_type]

In [ ]:
train_coco_path = '/Data_large/marine/Datasets/VENuS/annotations/coarse/train.json'
val_coco_path = '/Data_large/marine/Datasets/VENuS/annotations/coarse/val.json'
test_coco_path = '/Data_large/marine/Datasets/VENuS/annotations/coarse/test.json'

train_coco = get_coco(train_coco_path)
val_coco = get_coco(val_coco_path)
test_coco = get_coco(test_coco_path)

In [ ]:
annotations = train_coco['annotations']

vessel_types = [x['vessel_type'] for x in annotations]

vessel_types = [remapper(x) if isinstance(x, str) else 4  for x in vessel_types ]

counter = {0:0, 1:0, 2:0, 3:0, 4:0}
for v in vessel_types:
    counter[v] += 1


In [ ]:
def adjust_coco_annotations(coco_file):
    """
    Adjusts the COCO annotations to update category information and remap annotations based on vessel type.

    Parameters:
    coco_file (dict): A dictionary representing the COCO file structure. It should contain 'categories' and 'annotations' keys.

    Returns:
    dict: The updated COCO file with adjusted categories and annotations.

    The function performs the following steps:
    1. Updates the 'categories' in the COCO file by creating new categories with 'id', 'name' (vessel type), and 'supercategory' ('Ship').
       The 'id' for each category is assigned based on its index in the input 'classes' list.
    2. Iterates over the 'annotations' in the COCO file and updates the 'category_id' for each annotation.
       If the 'vessel_type' in an annotation is a string, it uses the 'remapper' function to map the 'vessel_type' to a new 'category_id'.

    Note:
    - The 'classes' list and 'remapper' function should be defined outside of this function.
    - The 'remapper' function should take a vessel type string as input and return the corresponding category ID.

    Example usage:
    ```python
    coco_file = {
        'categories': [{'id': 1, 'name': 'Tanker', 'supercategory': 'Ship'}, ...],
        'annotations': [{'id': 1, 'vessel_type': 'Tanker', 'category_id': 1}, ...],
        ...
    }
    
    classes = ['Tanker', 'Cargo', 'Fishing', ...]
    
    def remapper(vessel_type):
        mapping = {'Tanker': 0, 'Cargo': 1, 'Fishing': 2, ...}
        return mapping.get(vessel_type, -1)

    updated_coco_file = adjust_coco_annotations(coco_file)
    ```
    """
    # Step 1)
    new_categories = [{
        'id': idx,
        'name': vessel_type,
        'supercategory': 'Ship',
    } for idx, vessel_type in enumerate(classes)]

    coco_file['categories'] = new_categories
    
    # Step 2)
    annotations = coco_file['annotations']
    for idx, ann in enumerate(annotations):
        vessel_type = ann['vessel_type']
        if isinstance(vessel_type, str):
            ann['category_id'] = remapper(ann['vessel_type'])
        else:
            ann['category_id'] = len(new_categories)-1
            
    return coco_file


In [ ]:
mtrain_coco = adjust_coco_annotations(train_coco)
mval_coco = adjust_coco_annotations(val_coco)
mtest_coco = adjust_coco_annotations(test_coco)

In [20]:
outfolder = '/Data_large/marine/Datasets/VENuS/annotations/coarse_classifier'

for data, filename in zip([mtrain_coco, mval_coco, mtest_coco],['train.json','val.json','test.json']):
    save_to_json(data=data, file_path=f'{outfolder}/{filename}')


Data successfully saved to /Data_large/marine/Datasets/VENuS/annotations/coarse_classifier/train.json
Data successfully saved to /Data_large/marine/Datasets/VENuS/annotations/coarse_classifier/val.json
Data successfully saved to /Data_large/marine/Datasets/VENuS/annotations/coarse_classifier/test.json
